# Generalização: viés, variância e regularização
### GCC1734 Inteligência Artificial · Companion da seção 4

Este notebook acompanha a seção 4 das notas de aula 07. A ideia central é tornar observáveis, por simulação, os conceitos que aparecem conceitualmente no texto:

- underfitting e overfitting;
- decomposição viés-variância;
- regularização L2 (Ridge) e L1 (Lasso);
- validação cruzada para seleção de hiperparâmetros;
- métricas de avaliação em regressão e classificação.

A simulação principal treina vários modelos sobre muitos conjuntos de treino amostrados da mesma distribuição. Isso permite estimar empiricamente viés e variância das previsões.


In [ ]:
# Configuração global
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

RNG = np.random.default_rng(42)
np.set_printoptions(precision=3, suppress=True)


---
## 1. Underfitting e overfitting em regressão

Vamos usar uma função verdadeira não linear:

\[
f(x) = \sin(2\pi x)
\]

Os dados observados têm ruído:

\[
y = f(x) + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma^2)
\]

Nenhum modelo consegue remover o ruído irredutível. O que ele pode controlar é o equilíbrio entre erro sistemático (viés) e sensibilidade à amostra de treino (variância).


In [ ]:
def f_true(x):
    return np.sin(2 * np.pi * x)


def sample_regression_data(n=25, noise=0.25, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    x = rng.uniform(0, 1, size=n)
    y = f_true(x) + rng.normal(0, noise, size=n)
    return x.reshape(-1, 1), y


x_grid = np.linspace(0, 1, 300).reshape(-1, 1)
y_grid_true = f_true(x_grid.ravel())

X_demo, y_demo = sample_regression_data(n=25, noise=0.25, rng=RNG)

degrees = [0, 1, 3, 12]
fig, axes = plt.subplots(1, len(degrees), figsize=(14, 3.5), sharey=True)

for ax, degree in zip(axes, degrees):
    if degree == 0:
        model = make_pipeline(PolynomialFeatures(degree=0), LinearRegression())
        title = 'grau 0 - underfitting'
    elif degree == 12:
        model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
        title = 'grau 12 - overfitting'
    else:
        model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
        title = f'grau {degree}'

    model.fit(X_demo, y_demo)
    ax.scatter(X_demo.ravel(), y_demo, s=35, alpha=0.8, label='treino')
    ax.plot(x_grid.ravel(), y_grid_true, color='black', lw=2, label='f(x) verdadeira')
    ax.plot(x_grid.ravel(), model.predict(x_grid), lw=2, label='modelo')
    ax.set_title(title)
    ax.set_xlabel('x')

axes[0].set_ylabel('y')
axes[-1].legend(loc='lower right')
plt.tight_layout()
plt.show()


### Curvas de treino e validação

O erro de treino tende a cair quando aumentamos a complexidade. O erro de validação costuma ter formato de U: melhora até certo ponto e depois piora, porque o modelo passa a ajustar ruído.


In [ ]:
X, y = sample_regression_data(n=90, noise=0.25, rng=np.random.default_rng(7))
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.35, random_state=42)

rows = []
for degree in range(0, 16):
    model = make_pipeline(PolynomialFeatures(degree=degree), LinearRegression())
    model.fit(X_train, y_train)
    rows.append(
        {
            'grau': degree,
            'RMSE treino': mean_squared_error(y_train, model.predict(X_train)) ** 0.5,
            'RMSE validação': mean_squared_error(y_val, model.predict(X_val)) ** 0.5,
        }
    )

curvas = pd.DataFrame(rows)
melhor_grau = curvas.loc[curvas['RMSE validação'].idxmin(), 'grau']

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(curvas['grau'], curvas['RMSE treino'], marker='o', label='treino')
ax.plot(curvas['grau'], curvas['RMSE validação'], marker='o', label='validação')
ax.axvline(melhor_grau, color='black', linestyle='--', label=f'melhor validação: grau {melhor_grau}')
ax.set(xlabel='complexidade do modelo (grau polinomial)', ylabel='RMSE', title='Underfitting, zona adequada e overfitting')
ax.legend()
plt.show()

curvas


---
## 2. Simulação de viés e variância

A decomposição das notas compara o comportamento esperado do modelo sobre diferentes conjuntos de treinamento:

\[
E[(h(x)-y)^2] = 	ext{Viés}^2 + 	ext{Variância} + \sigma^2
\]

Como conhecemos a função verdadeira `f_true`, podemos estimar:

- **viés²:** média, ao longo de `x`, de `(média das previsões - f(x))²`;
- **variância:** média, ao longo de `x`, da variância das previsões entre diferentes amostras de treino;
- **erro esperado sem ruído novo:** média de `(previsão - f(x))²`.


In [ ]:
def make_poly_model(degree, kind='ols', alpha=1.0):
    if kind == 'ols':
        estimator = LinearRegression()
    elif kind == 'ridge':
        estimator = Ridge(alpha=alpha)
    elif kind == 'lasso':
        estimator = Lasso(alpha=alpha, max_iter=20_000)
    else:
        raise ValueError(kind)

    if degree == 0:
        return make_pipeline(PolynomialFeatures(degree=0), estimator)
    return make_pipeline(PolynomialFeatures(degree=degree, include_bias=False), StandardScaler(), estimator)


def bias_variance_experiment(configs, n_datasets=250, n_train=25, noise=0.25, seed=123):
    rng = np.random.default_rng(seed)
    predictions = {name: [] for name, _, _, _ in configs}

    for _ in range(n_datasets):
        X_train, y_train = sample_regression_data(n=n_train, noise=noise, rng=rng)
        for name, degree, kind, alpha in configs:
            model = make_poly_model(degree=degree, kind=kind, alpha=alpha)
            model.fit(X_train, y_train)
            predictions[name].append(model.predict(x_grid))

    summary = []
    pred_arrays = {}
    for name, _, _, _ in configs:
        pred = np.vstack(predictions[name])
        pred_arrays[name] = pred
        mean_pred = pred.mean(axis=0)
        bias2_x = (mean_pred - y_grid_true) ** 2
        var_x = pred.var(axis=0)
        mse_to_signal_x = ((pred - y_grid_true) ** 2).mean(axis=0)
        summary.append(
            {
                'modelo': name,
                'viés² médio': bias2_x.mean(),
                'variância média': var_x.mean(),
                'viés² + variância': bias2_x.mean() + var_x.mean(),
                'erro contra f(x)': mse_to_signal_x.mean(),
                'erro esperado com ruído': mse_to_signal_x.mean() + noise**2,
            }
        )

    return pd.DataFrame(summary), pred_arrays


configs = [
    ('constante grau 0', 0, 'ols', 1.0),
    ('linear grau 1', 1, 'ols', 1.0),
    ('polinomial grau 3', 3, 'ols', 1.0),
    ('polinomial grau 12', 12, 'ols', 1.0),
    ('ridge grau 12 alpha=1', 12, 'ridge', 1.0),
]

bv, pred_arrays = bias_variance_experiment(configs)
bv


In [ ]:
bv_long = bv.melt(
    id_vars='modelo',
    value_vars=['viés² médio', 'variância média'],
    var_name='componente',
    value_name='valor',
)

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=bv_long, x='modelo', y='valor', hue='componente', ax=ax)
ax.set(title='Estimativa empírica de viés² e variância', xlabel='', ylabel='valor médio')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, len(configs), figsize=(16, 3.3), sharey=True)

for ax, (name, _, _, _) in zip(axes, configs):
    pred = pred_arrays[name]
    mean_pred = pred.mean(axis=0)
    low, high = np.percentile(pred, [5, 95], axis=0)

    ax.plot(x_grid.ravel(), y_grid_true, color='black', lw=2, label='f(x)')
    ax.plot(x_grid.ravel(), mean_pred, color='tab:blue', lw=2, label='média das previsões')
    ax.fill_between(x_grid.ravel(), low, high, color='tab:blue', alpha=0.2, label='5%-95%')
    ax.set_title(name)
    ax.set_xlabel('x')

axes[0].set_ylabel('y')
axes[-1].legend(loc='lower right')
plt.tight_layout()
plt.show()


Leitura esperada:

- modelos muito simples têm alta componente de viés;
- modelos muito flexíveis têm alta variância;
- Ridge mantém a mesma família polinomial rica, mas reduz oscilações ao penalizar pesos grandes.


---
## 3. Regularização L2 e L1

A função de custo regularizada das notas é:

\[
J_{reg}(w) = J(w) + \lambda\Omega(w)
\]

Vamos começar reproduzindo o cálculo simples das notas para `w1=3.0`, `w2=0.1`, `J=1.5` e `lambda=0.5`.


In [ ]:
J = 1.5
lam = 0.5
w = np.array([3.0, 0.1])

pd.DataFrame(
    [
        {'penalidade': 'L2 / Ridge', 'Omega(w)': np.sum(w**2), 'J_regularizado': J + lam * np.sum(w**2)},
        {'penalidade': 'L1 / Lasso', 'Omega(w)': np.sum(np.abs(w)), 'J_regularizado': J + lam * np.sum(np.abs(w))},
    ]
)


### Ridge controla pesos grandes

No exemplo abaixo mantemos grau 12, mas variamos `alpha` do Ridge. Quanto maior `alpha`, maior a pressão para pesos menores. Regularização demais volta a produzir underfitting.


In [ ]:
alphas = [0.0, 0.001, 0.01, 0.1, 1, 10, 100]
rows = []

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
plot_alphas = [0.0, 1, 100]

for alpha in alphas:
    if alpha == 0.0:
        model = make_poly_model(12, kind='ols')
        label = 'OLS sem regularização'
    else:
        model = make_poly_model(12, kind='ridge', alpha=alpha)
        label = f'Ridge alpha={alpha}'
    model.fit(X_train, y_train)
    rmse_train = mean_squared_error(y_train, model.predict(X_train)) ** 0.5
    rmse_val = mean_squared_error(y_val, model.predict(X_val)) ** 0.5
    rows.append({'alpha': alpha, 'modelo': label, 'RMSE treino': rmse_train, 'RMSE validação': rmse_val})

for ax, alpha in zip(axes, plot_alphas):
    model = make_poly_model(12, kind='ols') if alpha == 0.0 else make_poly_model(12, kind='ridge', alpha=alpha)
    model.fit(X_train, y_train)
    ax.scatter(X_train.ravel(), y_train, s=25, alpha=0.7)
    ax.plot(x_grid.ravel(), y_grid_true, color='black', lw=2, label='f(x)')
    ax.plot(x_grid.ravel(), model.predict(x_grid), color='tab:red', lw=2, label='modelo')
    ax.set_title('sem reg.' if alpha == 0 else f'Ridge alpha={alpha}')
    ax.set_xlabel('x')

axes[0].set_ylabel('y')
axes[-1].legend()
plt.tight_layout()
plt.show()

pd.DataFrame(rows)


### Lasso pode zerar características irrelevantes

Agora criamos uma regressão com muitas características, mas só algumas realmente importam. Ridge tende a encolher todos os pesos; Lasso frequentemente zera vários deles, funcionando como seleção de características.


In [ ]:
rng = np.random.default_rng(11)
n = 180
p = 12
X_sparse = rng.normal(size=(n, p))
w_true = np.array([3.0, -2.0, 0.0, 0.0, 1.5, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0])
y_sparse = X_sparse @ w_true + rng.normal(0, 0.8, size=n)

Xtr, Xte, ytr, yte = train_test_split(X_sparse, y_sparse, test_size=0.3, random_state=42)

ridge = make_pipeline(StandardScaler(), Ridge(alpha=5.0))
lasso = make_pipeline(StandardScaler(), Lasso(alpha=0.12, max_iter=20_000))
ridge.fit(Xtr, ytr)
lasso.fit(Xtr, ytr)

coef = pd.DataFrame(
    {
        'feature': [f'x{j}' for j in range(p)],
        'peso verdadeiro': w_true,
        'Ridge': ridge.named_steps['ridge'].coef_,
        'Lasso': lasso.named_steps['lasso'].coef_,
    }
)

fig, ax = plt.subplots(figsize=(9, 4))
coef_plot = coef.melt(id_vars='feature', value_vars=['peso verdadeiro', 'Ridge', 'Lasso'], var_name='origem', value_name='peso')
sns.barplot(data=coef_plot, x='feature', y='peso', hue='origem', ax=ax)
ax.axhline(0, color='black', lw=1)
ax.set(title='Ridge encolhe pesos; Lasso tende a produzir esparsidade')
plt.tight_layout()
plt.show()

print('RMSE Ridge:', mean_squared_error(yte, ridge.predict(Xte)) ** 0.5)
print('RMSE Lasso:', mean_squared_error(yte, lasso.predict(Xte)) ** 0.5)
coef


---
## 4. Validação cruzada

Quando os dados são poucos, separar um conjunto de validação fixo pode desperdiçar exemplos. A validação cruzada k-fold estima desempenho médio treinando e validando em partições alternadas dos dados de treino.

Aqui selecionamos simultaneamente o grau polinomial e a força de regularização Ridge usando 5-fold CV. O conjunto de teste continua separado.


In [ ]:
X_all, y_all = sample_regression_data(n=120, noise=0.25, rng=np.random.default_rng(1234))
X_dev, X_test, y_dev, y_test = train_test_split(X_all, y_all, test_size=0.25, random_state=42)

cv = KFold(n_splits=5, shuffle=True, random_state=42)
rows = []

for degree in [1, 2, 3, 5, 8, 12]:
    for alpha in [0.001, 0.01, 0.1, 1, 10, 100]:
        model = make_poly_model(degree, kind='ridge', alpha=alpha)
        scores = cross_val_score(model, X_dev, y_dev, scoring='neg_root_mean_squared_error', cv=cv)
        rows.append(
            {
                'grau': degree,
                'alpha': alpha,
                'RMSE CV médio': -scores.mean(),
                'desvio CV': scores.std(),
            }
        )

cv_results = pd.DataFrame(rows).sort_values('RMSE CV médio')
best = cv_results.iloc[0]
final_model = make_poly_model(int(best['grau']), kind='ridge', alpha=float(best['alpha']))
final_model.fit(X_dev, y_dev)
test_rmse = mean_squared_error(y_test, final_model.predict(X_test)) ** 0.5

print(f"Melhor por CV: grau={int(best['grau'])}, alpha={best['alpha']}")
print(f"RMSE no teste final: {test_rmse:.3f}")
cv_results.head(10)


In [ ]:
pivot = cv_results.pivot(index='grau', columns='alpha', values='RMSE CV médio')
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis_r', ax=ax)
ax.set(title='RMSE médio por validação cruzada', xlabel='alpha Ridge', ylabel='grau polinomial')
plt.show()


---
## 5. Métricas de avaliação

A seção 4 também lembra que a métrica depende da tarefa. Em regressão, RMSE, MAE e R² respondem perguntas diferentes. Em classificação desbalanceada, acurácia pode enganar.


In [ ]:
# Métricas de regressão no modelo escolhido por CV.
y_pred_test = final_model.predict(X_test)

pd.DataFrame(
    [
        {
            'RMSE': mean_squared_error(y_test, y_pred_test) ** 0.5,
            'MAE': mean_absolute_error(y_test, y_pred_test),
            'R2': r2_score(y_test, y_pred_test),
        }
    ]
)


In [ ]:
# Classificação desbalanceada: um classificador trivial pode ter alta acurácia.
Xc, yc = make_classification(
    n_samples=1200,
    n_features=8,
    n_informative=4,
    n_redundant=1,
    weights=[0.94, 0.06],
    class_sep=0.9,
    flip_y=0.02,
    random_state=42,
)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.3, stratify=yc, random_state=42)

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
clf.fit(Xc_train, yc_train)

pred_model = clf.predict(Xc_test)
pred_trivial = np.zeros_like(yc_test)

metric_rows = []
for name, pred in [('sempre classe 0', pred_trivial), ('regressão logística', pred_model)]:
    metric_rows.append(
        {
            'modelo': name,
            'acurácia': accuracy_score(yc_test, pred),
            'precisão': precision_score(yc_test, pred, zero_division=0),
            'revocação': recall_score(yc_test, pred, zero_division=0),
            'F1': f1_score(yc_test, pred, zero_division=0),
        }
    )

metrics_cls = pd.DataFrame(metric_rows)
display(metrics_cls)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, (name, pred) in zip(axes, [('sempre classe 0', pred_trivial), ('regressão logística', pred_model)]):
    ConfusionMatrixDisplay(confusion_matrix(yc_test, pred)).plot(ax=ax, colorbar=False, values_format='d')
    ax.set_title(name)
plt.tight_layout()
plt.show()


## Fechamento para sala

Sugestões de perguntas para conduzir a discussão:

1. Qual modelo da simulação tem maior viés? Qual tem maior variância?
2. Por que o erro de treino cai mesmo quando o erro de validação sobe?
3. Ridge muda o espaço de hipóteses ou muda a preferência entre hipóteses dentro dele?
4. Em que situação Lasso é mais interessante que Ridge?
5. Por que validação cruzada não substitui o conjunto de teste?
6. Em dados desbalanceados, que erro fica escondido quando olhamos só a acurácia?
